### Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [ ]:
# Original library versions
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Preferred versions for Colab as of October 2025 (thanks, Lev!)
%pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
from transformers import BitsAndBytesConfig
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
model_name = "Enoch/llama-7b-hf"
model_path = "llama-7b-hf"
# Конфигурация для 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
try:
    # loading Llama tokenizer ...
    tokenizer = transformers.LlamaTokenizer.from_pretrained(
        model_path,
        device_map=device,
        local_files_only=True)
    # ... and the model itself
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map='auto',
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        # load_in_4bit=True,  deprecated
        quantization_config=quantization_config,
        torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
        local_files_only=True
    )
except:
    # loading Llama tokenizer ...
    tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
    # ... and the model itself
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        # load_in_4bit=True,  deprecated
        quantization_config=quantization_config,
        torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
    )
tokenizer.pad_token_id = tokenizer.eos_token_id

for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


In [1]:
# Очищаем кэш для этой модели (если требуется)
# import os
# cache_path = r"C:\Users\Дмитрий\.cache\huggingface\hub\models--Enoch--llama-7b-hf"
# if os.path.exists(cache_path):
#     import shutil
#     shutil.rmtree(cache_path)

### Prompt tuning: the story of a fox (1 point)
![img](fox_jump.png)
![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [4]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



Output: <s>A quick brown fox jumps over the lazy dog.
A quick


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [5]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.0562, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](prompt_tuning.png)
![img](https://i.imgur.com/VwNNKnb.png)


In [18]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace the model's original word embeddings with a layer - THIS layer
    - that inserts trainable prompts instead of the first N token embeddings.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Don't forget to prepend several BOS tokens to input_ids"
        )


        # Embed the input_ids using the original word embeddings
        # <YOUR CODE HERE>
        input_embeddings = self.original_word_embeddings(input_ids)  # Shape: [batch_size, seq_length, embedding_dim]

        # Replace the first num_prompts token embeddings with the learnable prompts
        batch_size = input_ids.shape[0]
        # <YOUR CODE HERE>
        learnable_prompts_expanded = self.learnable_prompts.repeat(batch_size, 1, 1)  # Shape: [batch_size, num_prompts, embedding_dim]
        # <YOUR CODE HERE>
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]  # Shape: [batch_size, seq_length - num_prompts, embedding_dim]

        # Concatenate learnable prompts with the embeddings of the remaining tokens
        # <YOUR CODE HERE>
        output_embeddings = torch.cat([learnable_prompts_expanded, remaining_embeddings], dim=1)

        return output_embeddings


In [20]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.amp.autocast('cuda', dtype=torch.float32):
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [26]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [30]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
num_epochs = 50
for epoch in range(num_epochs):
    # Подготовим данные с паддингами в начале
    batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
    space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                                   dtype=torch.int64, device=device)
    batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
    batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d}, Loss: {loss.item():.4f}")

        # Также посмотрим на норму промптов
        prompt_norm = torch.norm(model.model.embed_tokens.learnable_prompts).item()
        print(f"Prompt norm: {prompt_norm:.4f}")

# raise NotImplemented("Your task: iteratively train the model to reduce loss using prompt optimizer (opt)")

Epoch    0, Loss: 0.0055
Prompt norm: 260.2822
Epoch   10, Loss: 0.0048
Prompt norm: 260.2845
Epoch   20, Loss: 0.0043
Prompt norm: 260.2855
Epoch   30, Loss: 0.0039
Prompt norm: 260.2858
Epoch   40, Loss: 0.0036
Prompt norm: 260.2859


Первый раз обучал на 100 эпохах, увидел, что хватило и 50 эпох.

In [31]:
# Final loss assertion
assert loss.item() <= 0.1
print("Good job!")

Good job!


In [32]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Using HuggingFace PEFT (2 point)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [3]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [4]:
# Define the ground truth sentence
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [22]:
next_word_logits.size()

torch.Size([1, 22, 32000])

In [25]:
# Training Configuration
loss_threshold = 0.1  # Desired loss threshold
num_epochs = 30  # Max number of epochs
learning_rate = 1e-3  # <YOUR CODE HERE>  Learning rate

# Define the optimizer for trainable parameters (PEFT prompts)
optimizer =  torch.optim.Adam(model.parameters(), lr=learning_rate, fused=True)  # <YOUR CODE HERE> fused=True даёт ускорение, нововведение в Torch

# Define the ground truth sentence
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

# Training Loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(**batch)  # <YOUR CODE HERE>

    # Skip logits for virtual tokens and the last token
    next_word_logits = outputs.logits[:, peft_config.num_virtual_tokens : -1, :]  # <YOUR CODE HERE>  # Skip virtual tokens
    true_next_tokens = batch['input_ids'][:, 1:]  # <YOUR CODE HERE>  # Shift ground truth tokens by one

    # Compute the loss
    loss = F.cross_entropy(
        next_word_logits.flatten(0, 1),
        true_next_tokens.flatten(0, 1)
    )

    # Backpropagation
    # <YOUR CODE HERE>
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss for tracking
    if epoch % 10 == 9:
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

    # Stop training if loss is below threshold
    if loss.item() < loss_threshold:
        print("Loss threshold reached. Stopping training.")
        break
else:
    print("Maximum epochs reached without meeting the loss threshold.")

Loss threshold reached. Stopping training.


За 200-210 эпох обучилась

In [26]:
# Final assertion to ensure loss is below threshold
assert loss.item() < loss_threshold, "Training failed to reduce loss below threshold."
print("Training successful! Loss is below 0.1.")

Training successful! Loss is below 0.1.


In [27]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Generate 18 tokens
for i in range(15):
    # Forward pass to get the logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Append the next token to input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Update the attention_mask to match the new input_ids length
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Decode the generated sequence
# Skip the virtual tokens (if applicable) by slicing `batch["input_ids"][:, num_prompts:]`
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nOutput:", decoded_output)



Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
![img](adapter.png)
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [3]:
# re-load the model to remove any previous PEFT tuners
try:
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map='auto',
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        # load_in_4bit=True,  deprecated
        quantization_config=quantization_config,
        torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
        local_files_only=True
    )
except:
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        # load_in_4bit=True,  deprecated
        quantization_config=quantization_config,
        torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
    )

for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad

Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [4]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        original_output = self.module(input)  # [batch_size, ..., out_features] <YOUR CODE HERE>

        lora_output = input @ self.adapter_A @ self.adapter_B  # [batch_size, ..., out_features] <YOUR CODE HERE>

        return original_output + lora_output

In [5]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

In [6]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B

In [9]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
# test a single training step, make sure we get meaningful gradients
with torch.amp.autocast('cuda', dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [ ]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=1, # for effectively larger batch size
        warmup_steps=250, 
        max_steps=100, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs', 
        report_to=None,
        save_strategy="no" # to make it work as of October 2025 (thanks, Vlad!) 
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,1.891200
2,1.696000
3,0.896900
4,1.744600
5,1.168000
6,0.730000
7,1.525700
8,1.063700
9,0.669600
10,1.427400


TrainOutput(global_step=100, training_loss=0.5410390722751618, metrics={'train_runtime': 150.7473, 'train_samples_per_second': 1.327, 'train_steps_per_second': 0.663, 'total_flos': 621258424123392.0, 'train_loss': 0.5410390722751618, 'epoch': 6.25})

### Final task: *actually* train the model (5 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter train subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [31]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch', 'SELECT', 'void']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)
# checking if the model can learn. Change max_steps for proper training

In [17]:
def preprocess_function(samples):
    # Берем первые 512 символов каждой строки
    samples['content'] = [text[:512] for text in samples['content']]
    # токенизируем
    output = tokenizer(samples['content'])
    return output

In [40]:
import datasets
# Загружаем и предобрабатываем данные
data = datasets.load_dataset("codeparrot/codeparrot-clean", split="train[:1000]")

dataset = data.map(preprocess_function, batched=True, remove_columns=data.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [42]:
dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1000
})

In [27]:
# Добавляем LoRA адаптеры к дополнительным слоям
for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        # MLP компоненты
        module.mlp.gate_proj = LoRALayer(module.mlp.gate_proj, rank=lora_rank).to(device)
        module.mlp.up_proj = LoRALayer(module.mlp.up_proj, rank=lora_rank).to(device)
        module.mlp.down_proj = LoRALayer(module.mlp.down_proj, rank=lora_rank).to(device)

In [28]:
# Проверяем, сколько параметров обучается
def count_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Обучаемые параметры: {trainable_params:,}")
    print(f"Всего параметров: {total_params:,}")
    print(f"Процент обучаемых: {100 * trainable_params / total_params:.2f}%")

count_trainable_parameters(model)

Обучаемые параметры: 17,891,328
Всего параметров: 3,518,304,256
Процент обучаемых: 0.51%


In [30]:
# Функция генерации
def generate_python_code(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            do_sample=True
        )

    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_code

In [32]:
# Сгенерируем примеры до обучения
before_finetuning = []
for prompt in prompts:
    before_finetuning.append(generate_python_code(prompt))
before_finetuning

['100 Years of Celebrations\nThe story of how we got to where we are today!\nThe history of the Town of Colonial Beach, Virginia began in 1607 when Captain John Smith of Jamestown, Virginia, was exploring the Chesapeake Bay for its potential for trade and settlement. In 1608, he sailed up the Potomac River and noted the mouth of the river, the Rappahann',
 'import os\nimport sys\n\nfrom collections import OrderedDict\nfrom pathlib import Path\n\nfrom cwd import Cwd\n\n\nclass UserError(Exception):\n    pass\n\n\nclass CommandNotFoundError(UserError):\n    pass\n\n\nclass UserErrorHandler(object):\n    @staticmethod\n    def error_handler(err):\n        """\n        Internal: catch all errors and print the command and the path it was called from',
 'from __future__ import absolute_import, unicode_literals\n\nfrom django.contrib.auth.models import User\nfrom django.core.urlresolvers import reverse\nfrom django.template import Context, Template\nfrom django.utils import timezone\nfrom dja

In [43]:
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=dataset,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4, # for effectively larger batch size
        warmup_steps=250,
        max_steps=500,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir='codeparrot-finetuned',
        overwrite_output_dir=True,
        report_to=None,
        save_steps=100,
        save_total_limit=2,
        save_strategy="no" # to make it work as of October 2025 (thanks, Vlad!)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

D:\YandexDisk-Dmitry\HSE\NLP\venv\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.233400
20,1.128900
30,1.142500
40,1.008200
50,1.160100
60,1.210900
70,1.036800
80,1.129600
90,0.926300
100,1.119600


TrainOutput(global_step=500, training_loss=0.932945650100708, metrics={'train_runtime': 1871.4766, 'train_samples_per_second': 1.069, 'train_steps_per_second': 0.267, 'total_flos': 1.299910847422464e+16, 'train_loss': 0.932945650100708, 'epoch': 2.0})

In [44]:
# Сгенерируем примеры после обучения
after_finetuning = []
for prompt in prompts:
    after_finetuning.append(generate_python_code(prompt))
after_finetuning

D:\YandexDisk-Dmitry\HSE\NLP\venv\Lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


['#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n#\n# This file is part of Ansible\n#\n# Ansible is free software: you can redistribute it and/or modify\n# it under the terms of the GNU General Public License as published by\n# the Free Software Foundation, either version 3 of the License, or\n# (at your option) any later version.\n#\n# Ansible',
 "import sys\nimport unittest\n\nfrom test import support\nfrom test.support import run_unittest\nfrom test.test_urllib2 import test_get_all\nfrom test.test_urllib2 import test_parse_url\n\n\nclass UrlEncodeTests(unittest.TestCase):\n    def test_url_encode(self):\n        self.assertEqual(urllib.parse.urlencode({}), '{}')\n",
 "from __future__ import print_function\nimport os\nimport sys\nimport unittest\n\nimport pytest\nimport pkg_resources\ntry:\n    import mock\nexcept ImportError:\n    mock = None\n\ntest_resources = pkg_resources.DistributionDiscoverer('test_resources')\ntest_resources.find(self=True)\n\n\nclass DistributionCollectionT

In [48]:
from IPython.display import HTML, display

table_template = """<table style="border:1px solid black; width: 100%; table-layout: fixed;" >
  <tr>
    <th style="text-align: center; border:1px solid black; width: 20%">PROMPT</th>
    <th style="text-align: center; border:1px solid black; width: 40%">BEFORE</th>
    <th style="text-align: center; border:1px solid black; width: 40%">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black; word-wrap: break-word;"><pre align="left" style="white-space: pre-wrap; word-wrap: break-word;">`{}`</pre></td>
    <td style="width:40%; border:1px solid black; word-wrap: break-word;"><pre align="left" style="white-space: pre-wrap; word-wrap: break-word; max-height: 300px; overflow-y: auto;">{}</pre></td>
    <td style="width:40%; border:1px solid black; word-wrap: break-word;"><pre align="left" style="white-space: pre-wrap; word-wrap: break-word; max-height: 300px; overflow-y: auto;">{}</pre></td>
  </tr>'''

rows = []

for prompt, before, after in zip(prompts, before_finetuning, after_finetuning):
    rows.append(row_template.format(prompt, before, after))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,"100 Years of Celebrations The story of how we got to where we are today! The history of the Town of Colonial Beach, Virginia began in 1607 when Captain John Smith of Jamestown, Virginia, was exploring the Chesapeake Bay for its potential for trade and settlement. In 1608, he sailed up the Potomac River and noted the mouth of the river, the Rappahann","#!/usr/bin/env python # -*- coding: utf-8 -*- # # This file is part of Ansible # # Ansible is free software: you can redistribute it and/or modify # it under the terms of the GNU General Public License as published by # the Free Software Foundation, either version 3 of the License, or # (at your option) any later version. # # Ansible"
`import`,"import os import sys from collections import OrderedDict from pathlib import Path from cwd import Cwd class UserError(Exception): pass class CommandNotFoundError(UserError): pass class UserErrorHandler(object): @staticmethod def error_handler(err): """""" Internal: catch all errors and print the command and the path it was called from","import sys import unittest from test import support from test.support import run_unittest from test.test_urllib2 import test_get_all from test.test_urllib2 import test_parse_url class UrlEncodeTests(unittest.TestCase): def test_url_encode(self): self.assertEqual(urllib.parse.urlencode({}), '{}')"
`from`,"from __future__ import absolute_import, unicode_literals from django.contrib.auth.models import User from django.core.urlresolvers import reverse from django.template import Context, Template from django.utils import timezone from django.utils.encoding import iri_to_uri, smart_unicode # Import custom settings from kitsune.sumo.custom_settings import get_custom_settings settings =",from __future__ import print_function import os import sys import unittest import pytest import pkg_resources try: import mock except ImportError: mock = None test_resources = pkg_resources.DistributionDiscoverer('test_resources') test_resources.find(self=True) class DistributionCollectionTestCase(unittest.TestCase): def setUp
`while`,"while (1) do while (1) do something Post subject: while (1) do I am trying to get my head around this part of a C++ program. I can understand the part with the first if statement. I have tried searching for an explanation for the part with the second if statement, but can't find one. Can anyone explain what the second if statement does and why it is there? Post subject: Re: while (1",while True: line = raw_input() words = line.split() for word in words: if len(word) > 1 and word[0] == word[1]: print word \end{code} The problem is what happens when the user types: \begin{code} 1 2 3 \end{code} Is there a way to handle that?
`try`,"try: # pylint: disable=protected-access from collections import Iterable try: # pylint: enable=protected-access Iterable = collections.Iterable except AttributeError: Iterable = tuple # ------------------------------------------------------------------------- # Utility functions # ------------------------------------------------------------------------- def _sort_key(self, item): return item","try: import unittest2 as unittest except ImportError: import unittest from test import test_support import tempfile import time import sys import os import select import select import select import select import select import select class SelectTest(unittest.TestCase): def test_error(self): self.assertRaises(ValueError, select.select,"
`if`,if (window.console) window.console.log('[Jasmine] Loading the jasmine adapter'); // Load jQuery's jasmine adapter if (typeof window.jQuery === 'undefined' && typeof window.$ === 'undefined') { throw '[Jasmine] The jasmine adapter requires jQuery.' } // Create a global jasmine object window.jasmine = new jasmine.J,"if ( ! defined( 'ABSPATH' ) ) { define( 'ABSPATH', dirname( __FILE__ ) . '/' ); } if ( ! function_exists( 'add_theme_support' ) ) { /** * Adds support for a given feature. * * @since 1.0.0 *"
`for`,"for some reason

За пару эпох толком выучиться не успела, но всё равно виденпрогресс по сгенерированному тексту, он больше похож на код, чем до обучения.

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.